In [1]:
# %load_ext autoreload
# %autoreload 2

import sys
import pathlib

# Ensure the notebook can import the local src package from the project root.
project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sunpy.net import Fido, attrs as a
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
import sunpy.map


from src.utilities import make_cube

# Dowloading AR's
 Here i going to dowload the folowing AR's witch wilson depresions were already computed with a more sofisticated method ussing hinode telescope

| NOAA | Date | θ [deg] | Spot area [Mm²] | B_av [G] | z_W,div [km] | z_W,press [km] |
| --- | --- | --- | --- | --- | --- | --- |
| 11039 | 2010.01.01 | 29 | 284 | 2471 | 659 | 366 |
| 11041 | 2010.01.26 | 20 | 170 | 2068 | 638 | 305 |
| 11106 | 2010.09.16 | 27 | 195 | 2349 | 636 | 381 |
| 11117 | 2010.10.27 | 25 | 522 | 2093 | 561 | 291 |
| 11117 | 2010.10.28 | 35 | 223 | 2090 | 625 | 313 |
| 11363 | 2011.12.06 | 25 | 1268 | 2287 | 524 | 326 |
| 11536 | 2012.07.31 | 34 | 124 | 2181 | 577 | 346 |

## Locating each AR

`a.Instrument.noaa_indices` only gives the global daily sunspot number — it has no per-region position. To get the position of a specific NOAA AR we query the **HEK** (Heliophysics Event Knowledgebase) for `AR` events filtered by `a.hek.AR.NOAANum`. Each matching record carries the region's helioprojective bounding box (`hpc_bbox`) at that date, which we turn into a `(bottom_left, top_right)` pair for a JSOC cutout request.

Since we now want several continuous days per AR (not a single-day snapshot), `locate_ar_window` (in `src/utilities.py`) queries HEK across the *whole* `N_DAYS` window: it takes the **largest** reported half-width/half-height across all those days (the AR's real size can change noticeably day to day) but anchors the box's **position** at the first day only — position drift across the window is left entirely to `a.jsoc.Cutout(tracking=True)`, which follows solar rotation automatically. This keeps every output frame the same pixel shape (required for `make_cube`'s stacking) while staying large enough to contain the AR for the full window.

In [2]:
active_regions = [
    {'noaa': 11039, 'date': '2010-01-01'},
    {'noaa': 11041, 'date': '2010-01-26'},
    {'noaa': 11106, 'date': '2010-09-16'},
    # 11117's two catalog dates (2010-10-27, 2010-10-28) are one continuous AR passage —
    # a multi-day window anchored at the earlier date already covers the later one, so
    # they're merged into a single entry instead of being processed twice.
    {'noaa': 11117, 'date': '2010-10-27'},
    {'noaa': 11363, 'date': '2011-12-06'},
    {'noaa': 11536, 'date': '2012-07-31'},
]

In [3]:
from src.utilities import locate_ar_window

N_DAYS = 2  # minimum continuous days of data to fetch per AR

for ar in active_regions:
    ar['bl'], ar['tr'], ar['time_start'], ar['time_end'] = locate_ar_window(ar['noaa'], ar['date'], n_days=N_DAYS)
    print(f"NOAA {ar['noaa']} ({ar['date']}): window {ar['time_start']} .. {ar['time_end']}  "
          f"bl={ar['bl'].Tx:.1f},{ar['bl'].Ty:.1f}  tr={ar['tr'].Tx:.1f},{ar['tr'].Ty:.1f}")

NOAA 11039 (2010-01-01): window 2010-01-01 00:00:00 .. 2010-01-02 23:59:59  bl=223.4 arcsec,-451.3 arcsec  tr=396.7 arcsec,-382.8 arcsec
NOAA 11041 (2010-01-26): window 2010-01-26 00:00:00 .. 2010-01-27 23:59:59  bl=-297.7 arcsec,-360.6 arcsec  tr=-71.2 arcsec,-295.2 arcsec
NOAA 11106 (2010-09-16): window 2010-09-16 00:00:00 .. 2010-09-17 23:59:59  bl=-570.3 arcsec,-561.1 arcsec  tr=13.9 arcsec,-303.6 arcsec
NOAA 11117 (2010-10-27): window 2010-10-27 00:00:00 .. 2010-10-28 23:59:59  bl=65.4 arcsec,126.6 arcsec  tr=408.9 arcsec,402.6 arcsec
NOAA 11363 (2011-12-06): window 2011-12-06 00:00:00 .. 2011-12-07 23:59:59  bl=-225.0 arcsec,-511.5 arcsec  tr=510.7 arcsec,-199.3 arcsec
NOAA 11536 (2012-07-31): window 2012-07-31 00:00:00 .. 2012-08-01 23:59:59  bl=-481.2 arcsec,-481.8 arcsec  tr=-284.0 arcsec,-412.0 arcsec


## Downloading tracked cutouts from JSOC

With each AR's `(bottom_left, top_right)` box and full `time_start`/`time_end` window in hand, we reuse `src/scripts.py`'s `download_regions`/`make_cubes` — the same helpers already used for the DS9-defined regions — but driven by the HEK-derived boxes instead. `a.jsoc.Cutout(..., tracking=True)` makes JSOC do the cropping and solar-rotation tracking server-side, so we only ever download the small cutout, not the full 4096x4096 disk.

Each AR gets its own `data/raw/NOAA_<number>_<date>/region_01/` directory. `download_regions` now also **resumes** instead of re-fetching: it checks the timestamps already on disk for a region/series and only requests the missing tail of the window, so re-running this notebook after an interrupted download won't duplicate files or waste JSOC export requests.

**Note:** HMI science data only starts ~2010-05-01, so AR 11039 (2010-01-01) and AR 11041 (2010-01-26) predate it and will return zero results — that's expected, not a bug.

**Data volume:** NOAA 11117 is forced onto `hmi.*_45s` for its *entire* `N_DAYS`-day window (see `series_override` below) because JSOC's `hmi.*_720s` series 500-errors for any query touching 2010-10-26..29 — a JSOC-side gap, not our bug. Native 45s cadence over multiple days is an order of magnitude more data than everyone else's 720s. To keep that in check, `sample_override` applies `a.Sample(360*u.s)` for 11117 — JSOC downsamples **server-side**, so we only ever download every 8th 45s frame (≈360s cadence), roughly matching the other ARs' 720s data volume instead of dwarfing it.

In [4]:
from src.scripts import download_regions, make_cubes

notify_email = 'thomas.quamtum@gmail.com'  # must be a JSOC-registered export email
series_list  = ['hmi.Ic_720s', 'hmi.V_720s', 'hmi.M_720s']

# JSOC's *_720s series 500-errors for any query touching 2010-10-26..29 (a gap/bug in their
# 720s index for that window) even though the underlying 45s data is fine. NOAA 11117's
# whole N_DAYS window falls in/around that gap, so fetch it at 45s cadence instead.
series_override = {
    11117: ['hmi.Ic_45s', 'hmi.V_45s', 'hmi.M_45s'],
}

# 45s cadence over N_DAYS days is far more data than everyone else's 720s — downsample
# NOAA 11117 server-side to ~360s (every 8th 45s frame) to keep its volume comparable.
sample_override = {
    11117: 360 * u.s,
}

In [5]:
dry_run = False

all_summaries = []  # (noaa, date, series, summary) for the consolidated report below

for ar in active_regions:
    region_base = pathlib.Path('../data/raw') / f"NOAA_{ar['noaa']}_{ar['date']}"
    regions_hpc = [(ar['bl'], ar['tr'])]
    ar_series   = series_override.get(ar['noaa'], series_list)
    ar_sample   = sample_override.get(ar['noaa'])

    if dry_run:
        print(f"[dry run] NOAA {ar['noaa']} ({ar['date']}) -> {region_base} "
              f"({ar['time_start']} .. {ar['time_end']}, {ar_series}, sample={ar_sample})")
        continue

    for series in ar_series:
        summary = download_regions(regions_hpc, region_base, ar['time_start'], ar['time_end'],
                                    notify_email, series, sample=ar_sample)
        all_summaries.append((ar['noaa'], ar['date'], series, summary))

# Consolidated pass/fail report — useful after a long multi-AR, multi-day run.
failed = [(noaa, date, series, region, info)
          for noaa, date, series, summary in all_summaries
          for region, info in summary.items() if info['status'] == 'failed']
if failed:
    print("\nFAILED downloads:")
    for noaa, date, series, region, info in failed:
        print(f"  NOAA {noaa} ({date}) region_{region:02d} [{series}]: {info['error']}")
else:
    print("\nNo failures." if all_summaries else "")

Region 01 [hmi.Ic_720s]: 0 new frames found


Files Downloaded: 0file [00:00, ?file/s]

  -> 0 files saved to ../data/raw/NOAA_11039_2010-01-01/region_01
Region 01 [hmi.V_720s]: 0 new frames found


Files Downloaded: 0file [00:00, ?file/s]

  -> 0 files saved to ../data/raw/NOAA_11039_2010-01-01/region_01
Region 01 [hmi.M_720s]: 0 new frames found


...

hmi.m_720s.20120801_232400_TAI.1.magnetogram.fits:   0%|          | 0.00/86.4k [00:00<?, ?B/s]

hmi.m_720s.20120801_233600_TAI.1.magnetogram.fits:   0%|          | 0.00/86.4k [00:00<?, ?B/s]

hmi.m_720s.20120801_234800_TAI.1.magnetogram.fits:   0%|          | 0.00/86.4k [00:00<?, ?B/s]

hmi.m_720s.20120802_000000_TAI.1.magnetogram.fits:   0%|          | 0.00/86.4k [00:00<?, ?B/s]

  -> 120 files saved to ../data/raw/NOAA_11536_2012-07-31/region_01

No failures.


In [6]:
# 11039/11041 predate HMI science data (starts ~2010-05-01) — genuinely no data to fetch.
# 11117 is kept: it's handled via series_override (45s instead of the broken 720s window).
exclude_noaa = [11039, 11041]
active_regions = [ar for ar in active_regions if ar['noaa'] not in exclude_noaa]

In [7]:
ar = active_regions[2]  
region_base = pathlib.Path('../data/raw') / f"NOAA_{ar['noaa']}_{ar['date']}"
regions_hpc = [(ar['bl'], ar['tr'])]
for series in series_override.get(ar['noaa'], series_list):
    make_cubes(regions_hpc, region_base, series)

2026-08-06 16:30:00 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.


make_cube: reprojected 61/179 mismatched-shape frame(s) onto the majority (620, 1459) grid (../data/raw/NOAA_11363_2011-12-06/region_01/hmi.ic_720s.*.continuum.fits)


2026-08-06 16:31:21 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.


make_cube: reprojected 61/179 mismatched-shape frame(s) onto the majority (620, 1459) grid (../data/raw/NOAA_11363_2011-12-06/region_01/hmi.v_720s.*.fits)


2026-08-06 16:32:38 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.


make_cube: reprojected 60/178 mismatched-shape frame(s) onto the majority (620, 1459) grid (../data/raw/NOAA_11363_2011-12-06/region_01/hmi.m_720s.*.magnetogram.fits)


In [8]:
if True:
    for ar in active_regions:
        region_base = pathlib.Path('../data/raw') / f"NOAA_{ar['noaa']}_{ar['date']}"
        regions_hpc = [(ar['bl'], ar['tr'])]
        for series in series_override.get(ar['noaa'], series_list):
            make_cubes(regions_hpc, region_base, series)

2026-08-06 16:33:29 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.


make_cube: reprojected 60/173 mismatched-shape frame(s) onto the majority (511, 1159) grid (../data/raw/NOAA_11106_2010-09-16/region_01/hmi.ic_720s.*.continuum.fits)


2026-08-06 16:34:23 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.


make_cube: reprojected 61/174 mismatched-shape frame(s) onto the majority (511, 1159) grid (../data/raw/NOAA_11106_2010-09-16/region_01/hmi.v_720s.*.fits)


2026-08-06 16:35:20 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.


...

2026-08-06 16:40:20 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.


make_cube: reprojected 61/179 mismatched-shape frame(s) onto the majority (620, 1459) grid (../data/raw/NOAA_11363_2011-12-06/region_01/hmi.v_720s.*.fits)


2026-08-06 16:41:28 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.


make_cube: reprojected 60/178 mismatched-shape frame(s) onto the majority (620, 1459) grid (../data/raw/NOAA_11363_2011-12-06/region_01/hmi.m_720s.*.magnetogram.fits)


2026-08-06 16:41:30 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.
2026-08-06 16:41:32 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.
2026-08-06 16:41:35 - astropy - WARNING: VerifyWarning: Invalid 'BLANK' keyword in header.  The 'BLANK' keyword is only applicable to integer data, and will be ignored in this HDU.
